# 2.5D unstructured LES solver on an A100: cylinder Re 3900, shear-layer-refined mesh (reviewer item C)

V3 passed on all four reference quantities (St, C_D, C_pb, L_r; record section 65) on a 60k-cell mesh, but the peak centreline turbulence intensity ran 25-30% low against Parnaudeau's PIV/LES (section 65a): the diagnosis is under-resolved separating shear layers, not a modelling error. This notebook re-runs the same case on a mesh refined in the ring (wall cell ~0.002 D, arc 0.0073 D, ~1.7x the cell count: 104k vs 60k) and near wake (0.04 -> ~0.028 D cells), **warm-started by interpolating the finished V3 field onto the new mesh** (`interpolate_cylinder3900_ic.py`) rather than an impulsive start from rest -- this skips the ~25 D/U transient and the WALE-delay dance the original run needed (record section 64), so a much shorter run should reach the same statistical quality.

The success criterion (reviewer item C): the refined mesh's peak centreline `<u'u'>` should rise toward the 0.10-0.12 PIV/LES band (was 0.084 on the coarse mesh), confirming the gap was resolution, not the SGS model -- everything else (St, C_D, C_pb, L_r) should hold at their already-passing values.


In [ ]:
# 0a. Find the repository root (the notebook lives in tools/a100/, and Jupyter's working directory is
#     usually the notebook's own folder), or clone the repo if the notebook was downloaded on its own.
import os, subprocess
def find_root():
    d = os.getcwd()
    for _ in range(4):
        if os.path.exists(os.path.join(d, "src", "upiso25.py")): return d
        d = os.path.dirname(d)
    return None
root = find_root()
if root is None:
    print("repository not found above this directory; cloning the unstructured branch ...")
    subprocess.check_call(["git", "clone", "-q", "-b", "unstructured", "--depth", "1", "https://github.com/chandc/PICT-Python.git"]); root = os.path.abspath("PICT-Python")
os.chdir(root); print("working directory:", root)
# which version of the code this is: commit, branch, date, and whether the tree is clean. Compare with
# `git log -1 origin/unstructured` on GitHub; if behind, run  git pull  (or re-clone) and restart the kernel.
def git(*args):
    try: return subprocess.check_output(["git", *args], stderr=subprocess.DEVNULL).decode().strip()
    except Exception as e: return f"(git unavailable: {e})"
print("commit  :", git("log", "-1", "--format=%h %cd %s", "--date=short"))
print("branch  :", git("rev-parse", "--abbrev-ref", "HEAD"), "  dirty files:", len(git("status", "--porcelain").splitlines()))
remote = git("ls-remote", "https://github.com/chandc/PICT-Python.git", "refs/heads/unstructured").split()[:1]
head = git("rev-parse", "HEAD")
if remote and not head.startswith(remote[0]):
    # behind (or on another branch): bring this checkout to the tip of origin/unstructured
    print("checkout", head[:7], "is not origin/unstructured", remote[0][:7], "-> updating ...")
    print(git("fetch", "-q", "origin", "unstructured")); print(git("checkout", "-q", "-B", "unstructured", "FETCH_HEAD"))
    head = git("rev-parse", "HEAD"); print("now at  :", git("log", "-1", "--format=%h %cd %s", "--date=short"))
print("GitHub  :", remote[0][:7] if remote else "(offline)", "on origin/unstructured", "" if (remote and head.startswith(remote[0])) else "  <-- STILL different: delete the directory and re-run this cell to clone afresh")
assert os.path.exists("run_ucylinder3900.py") and os.path.exists("meshes/cylinder_re3900.msh"), "run_ucylinder3900.py / meshes/cylinder_re3900.msh missing: this is an old tree; delete it and re-run this cell"
print("driver  : run_ucylinder3900.py and meshes/cylinder_re3900.msh present  (ok)")
import importlib, sys as _s
for mod in [m for m in list(_s.modules) if m == "src" or m.startswith("src.")]: del _s.modules[mod]      # drop any solver modules a previous run imported from the old tree


## 0b. Google Drive: every output written there directly
Hosted runtimes lose their disk when the session ends. This mounts Drive (Colab) and sets `DRIVE_OUT = MyDrive/PICT-Python_runs/<tag>/`; the run cell passes it to the solver as `--outdir`, so the checkpoint, the final field, the statistics and the log are written to Drive as they are produced, not copied afterwards. `restore_from_drive()` and `sync_to_drive()` remain for figures and for older local files.

In [ ]:
import os, shutil, glob, threading, time
tag = "ucyl3900_cylinder_re3900_refined_nz64_wale"
DRIVE_ROOT = None
try:
    from google.colab import drive                       # Colab
    drive.mount("/content/drive", force_remount=False); DRIVE_ROOT = "/content/drive/MyDrive/PICT-Python_runs"
except Exception as e:
    for cand in (os.path.expanduser("~/Google Drive/My Drive"), "/content/drive/MyDrive"):
        if os.path.isdir(cand): DRIVE_ROOT = os.path.join(cand, "PICT-Python_runs"); break
    if DRIVE_ROOT is None: print("no Google Drive here (not Colab, no local Drive folder):", type(e).__name__, "- outputs stay local")
if DRIVE_ROOT:
    DRIVE_OUT = os.path.join(DRIVE_ROOT, tag); os.makedirs(DRIVE_OUT, exist_ok=True); os.makedirs(os.path.join(DRIVE_OUT, "logs"), exist_ok=True); print("Drive folder:", DRIVE_OUT)

def sync_to_drive(verbose=True):
    """copy this run's outputs to Drive if newer than the copy there; safe to call any time"""
    if not DRIVE_ROOT: return
    n = 0
    for pat, sub in ((f"results/{tag}*.npz", ""), (f"results/logs/{tag}*.log", "logs"), ("figures/*re3900*.png", ""), ("figures/*3900*.png", ""), ("results/*refined_ic*.npz", "")):
        for f in glob.glob(pat):
            dst = os.path.join(DRIVE_OUT, sub, os.path.basename(f))
            if not os.path.exists(dst) or os.path.getmtime(f) > os.path.getmtime(dst) + 1:
                shutil.copy2(f, dst); n += 1
    if verbose: print(time.strftime("%H:%M:%S"), f"synced {n} file(s) to Drive")

def _sync_loop(period=600):
    while True:
        time.sleep(period)
        try: sync_to_drive(verbose=False)
        except Exception as e: print("drive sync failed:", e)

def restore_from_drive():
    """bring a checkpoint/log back from Drive into results/ (new session, or after the local disk was lost)"""
    if not DRIVE_ROOT: return None
    os.makedirs("results/logs", exist_ok=True); got = []
    for f in glob.glob(os.path.join(DRIVE_OUT, f"{tag}*.npz")) + glob.glob(os.path.join(DRIVE_OUT, "logs", f"{tag}*.log")):
        dst = os.path.join("results", "logs" if f.endswith(".log") else "", os.path.basename(f))
        if not os.path.exists(dst): shutil.copy2(f, dst); got.append(dst)
    print("restored from Drive:", got or "nothing new"); return got
restore_from_drive()

In [ ]:
# 0. Dependencies into THIS kernel (pip of the running interpreter, not the shell's). pyamg builds the
#    multigrid hierarchy on the host; cupy-cuda12x is the GPU array library (binary wheels on x86-64).
import sys, subprocess, importlib
def ensure(mod, pkg):
    try: importlib.import_module(mod); print(f"ok  {mod}")
    except ImportError:
        print(f"installing {pkg} ..."); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg]); importlib.invalidate_caches(); importlib.import_module(mod); print(f"ok  {mod} (installed)")
ensure("numpy", "numpy>=1.26"); ensure("scipy", "scipy>=1.11"); ensure("pyamg", "pyamg>=5.0"); ensure("matplotlib", "matplotlib>=3.7")
# CuPy must match the DRIVER's CUDA version, not just be importable: a wheel built for CUDA 13 on a machine
# whose driver supports 12.x imports fine and then fails on the first array with
# "cudaErrorInsufficientDriver". Test a real device operation and reinstall the right wheel if needed.
import re
def cupy_works():
    try:
        import cupy; cupy.zeros(1).sum(); print("ok  cupy", cupy.__version__, "on", cupy.cuda.runtime.getDeviceProperties(0)["name"]); return True
    except Exception as e:
        print("cupy not usable:", type(e).__name__, str(e)[:160]); return False
if not cupy_works():
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout; m_ = re.search(r"CUDA Version: (\d+)\.(\d+)", smi)
    major = int(m_.group(1)) if m_ else 12; pkg = "cupy-cuda12x" if major < 13 else "cupy-cuda13x"
    print(f"driver supports CUDA {m_.group(0) if m_ else '?'} -> installing {pkg} (removing any other cupy build)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "cupy", "cupy-cuda12x", "cupy-cuda13x", "cupy-cuda11x"], capture_output=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    raise SystemExit(f"{pkg} installed. RESTART THE KERNEL/RUNTIME (Runtime > Restart session), then run the cells again from the top.")


## 0c. The refined mesh, and the warm-start initial condition

Builds `meshes/cylinder_re3900_refined.msh` if it is not already in the repo (it should be -- generated and committed locally; this is a fallback), copies the finished V3 run's field from Drive (`PICT-Python_runs/ucyl3900_cylinder_re3900_nz64_wale/..._final.npz`), and interpolates it onto the refined mesh's cell centroids (`interpolate_cylinder3900_ic.py`, linear per Fourier plane, ~1-2 min on the CPU, no GPU needed for this step).


In [ ]:
REFINED_MESH = "meshes/cylinder_re3900_refined.msh"
if not os.path.exists(REFINED_MESH):
    print("mesh not found in the checkout, generating it ...")
    subprocess.check_call([sys.executable, "meshes/make_cylinder_re3900.py", "--nt", "108", "--nr", "78", "--qr", "1.045", "--nv", "50", "--nw1", "160", "--nw2", "75", "--qw2", "1.035", "--name", "cylinder_re3900_refined"])

V3_TAG = "ucyl3900_cylinder_re3900_nz64_wale"
V3_FIELD_LOCAL = f"results/{V3_TAG}_final.npz"
os.makedirs("results", exist_ok=True)
if not os.path.exists(V3_FIELD_LOCAL):
    v3_drive = os.path.join(DRIVE_ROOT, V3_TAG, f"{V3_TAG}_final.npz") if DRIVE_ROOT else None
    assert v3_drive and os.path.exists(v3_drive), f"the finished V3 field is not on Drive at the expected path ({v3_drive}); check PICT-Python_runs/{V3_TAG}/ (DRIVE_ROOT={DRIVE_ROOT})"
    shutil.copy2(v3_drive, V3_FIELD_LOCAL); print("copied the V3 field from Drive:", v3_drive)
else:
    print("V3 field already local:", V3_FIELD_LOCAL)

IC = "results/ucyl3900_refined_ic.npz"
if not os.path.exists(IC):
    subprocess.check_call([sys.executable, "interpolate_cylinder3900_ic.py", "--src", V3_FIELD_LOCAL, "--src-mesh", "meshes/cylinder_re3900.msh", "--dst-mesh", REFINED_MESH, "--out", IC])
else:
    print("interpolated IC already built:", IC)


In [ ]:
import os, sys, subprocess, time; sys.path.insert(0, os.getcwd())      # cell 0a set the working directory to the repo root
import numpy as np, cupy as cp
p = cp.cuda.runtime.getDeviceProperties(0); print(p["name"], f"{p['totalGlobalMem']/2**30:.0f} GB, {p['multiProcessorCount']} SMs, cupy {cp.__version__}")
import pyamg, scipy; print("pyamg", pyamg.__version__, "scipy", scipy.__version__, "numpy", np.__version__)   # if this fails, rerun cell 0

## 1. Profile one step on this GPU, on the refined mesh
`meshes/cylinder_re3900_refined.msh`: ~104,000 quads (wall cell ~0.002 D, arc 0.0073 D vs the original 0.0109 D; near-wake cells ~0.028 D vs 0.04 D) x 64 Fourier planes. About 1.7x the original mesh's cell count; the original measured 620-745 ms/step on the A100 (record section 65), so expect roughly 1.0-1.3 s/step here (the pressure solve's iteration count may rise a little with the finer near-wall cells, profiled below rather than assumed).


In [ ]:
!{sys.executable} tools/a100/profile_step.py --mesh meshes/cylinder_re3900_refined.msh --nz 64 --steps 5


## 2. Correctness on this device (2 min)
The 3D Taylor–Green energy balance: −dE/dt must equal the scheme's own discrete dissipation to 0.1–0.2% at every sample (record section 51), E₀ = 31.006277.

In [ ]:
!TGV_DEVICE=gpu TGV_SOLVER=amg TGV_RE=100 TGV_T=2 TGV_N=32 TGV_NZ=32 TGV_DT=0.02 {sys.executable} test_utgv3d.py A

## 3. The cylinder at Re 3900, refined mesh, warm-started

Setup as the original V3 run (`run_ucylinder3900.py`) except: `--mesh meshes/cylinder_re3900_refined.msh`, `--init-field results/ucyl3900_refined_ic.npz` (the interpolated warm start built in 0c -- the solver's own lazy flux bootstrap, `init_flux()`, handles the cold `Ff` on step 1 exactly as it does for any fresh field, so no special handling is needed), and a short `--t-sgs 0.5` (the interpolated field is already a developed turbulent state, not an impulsive one, so the long delay the original run needed at `t_sgs 2` is not -- a short safety margin covers any small interpolation artifact right at the wall).

Because the warm start skips the original run's ~25 D/U instability-growth transient, a much shorter window should suffice: `--T 60 --t-stats 20` (40 D/U of statistics, ~8 shedding periods at St 0.21 -- matching or exceeding Kravchenko & Moin's own 7-period budget) is the default below; extend `--T` if the interim `_stats_partial.npz` at the last checkpoint still looks unsettled.


In [ ]:
# The run, in the FOREGROUND of this cell (a running cell keeps the session alive; a nohup'd process does not),
# writing every output straight to Drive (--outdir), and resuming automatically from the checkpoint there.
# If the session still drops, just run this cell again in the new session: it continues where the checkpoint is.
assert os.path.exists("run_ucylinder3900.py"), "old tree: re-run cell 0a"
OUT = DRIVE_OUT if DRIVE_ROOT else "results"; os.makedirs(os.path.join(OUT, "logs"), exist_ok=True)
ck = os.path.join(OUT, f"{tag}_ckpt.npz"); log_path = os.path.join(OUT, "logs", f"{tag}.log")
cmd = [sys.executable, "-u", "run_ucylinder3900.py", "--device", "gpu", "--mesh", "meshes/cylinder_re3900_refined.msh", "--nz", "64", "--dt", "0.004", "--cfl-max", "0.8", "--sgs", "wale", "--t-sgs", "0.5", "--init-field", IC,
       "--T", "60", "--t-stats", "20", "--report", "250", "--checkpoint", "2000", "--tag", tag, "--outdir", OUT] + (["--restart", ck] if os.path.exists(ck) else [])
print("resuming from", ck if os.path.exists(ck) else "scratch"); print(" ".join(cmd)); print("log:", log_path, flush=True)
with open(log_path, "a") as logf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:                       # stream: the cell shows progress live and the log on Drive grows line by line
        if "Warning" in line: continue
        print(line, end="", flush=True); logf.write(line); logf.flush()
    proc.wait(); print("exit code", proc.returncode)

## 4. Compare with the Re 3900 references
When the log shows `RESULT ...` (or from the interim `_stats_partial.npz` at any checkpoint after t = 50): force history and Strouhal number, C_p around the cylinder, the mean streamwise velocity on the wake centreline (recirculation length), and the cross-wake mean and rms profiles at x = 1.06, 1.54, 2.02 D (Parnaudeau's PIV stations).


In [ ]:
sf = os.path.join(OUT, f"{tag}_stats.npz"); sf = sf if os.path.exists(sf) else os.path.join(OUT, f"{tag}_stats_partial.npz")
print("statistics file:", sf)
!{sys.executable} plot_utility/plot_ucylinder_re3900.py "{sf}"
from IPython.display import Image, display
for f in ("figures/ucylinder_re3900_forces.png", "figures/ucylinder_re3900_wake.png", "figures/ucylinder_re3900_fields.png"):
    if os.path.exists(f): display(Image(f)); shutil.copy2(f, OUT)


## 5. Notes
* **Success criterion (reviewer item C):** peak centreline `<u'u'>` should rise from the coarse mesh's 0.084 toward the PIV/LES band 0.10-0.12 (record section 65a), while St, C_D, C_pb and L_r hold at their already-passing values (St 0.209, C_D 1.007, C_pb -0.913, L_r 1.53 D). If `<u'u'>` rises but St/C_D/L_r drift outside their bands, that is a real finding too -- report it, don't just take the peak-intensity number.
* If the run diverges near t = 0 despite the warm start, the interpolation likely left a small artifact right at the new, finer wall cells (the old and new near-wall layers don't align exactly); raise `--t-sgs` a little (1.0) or lower `--cfl-max` for the first checkpoint.
* `--nz 48` halves the spanwise cost if 60 D/U at 64 planes is too slow; keep `--T`/`--t-stats` as given.
* Compare directly against the coarse-mesh figures already in the repo: `figures/ucylinder_re3900_forces.png`, `_wake.png`, `_fields.png`, `_vs_piv.png` (record sections 65, 65a).
